In [125]:
import allel 
import numpy as np

import pandas as pd
from Bio import SeqIO

from plotnine import *
import os

In [126]:
minmaf = 0.05
samplefile = "samples.txt"
gapfile="{}_consensus_gaps.txt"

targets = ["RSVA","RSVB"]
#targetfile = ""

outsum = "ivar_maf_summary.txt"
outmaf = "ivar_maf.txt"

In [127]:

samples = [line.strip() for line in open(samplefile, 'r')]

#targets = [line.strip() for line in open(targetfile, 'r')]



In [128]:
allmaf = pd.DataFrame({'MAF':list(),
             'sample':list(),
             'target':list()})


In [130]:

for target in targets:
    gaps = pd.read_csv("{}_consensus_gaps.txt".format(target),sep="\t",index_col=False)

    for sample in samples:
        ivfile = "{}_{}_ivariants.tsv".format(sample,target)
        if not os.path.exists(ivfile): continue
        
        ivars = pd.read_csv(ivfile,sep="\t")
        
        ivars['MAF'] = ivars['ALT_FREQ']
        ivars.loc[(ivars['ALT_FREQ'] > 0.5),'MAF'] = 1-ivars.loc[(ivars['ALT_FREQ'] > 0.5),'ALT_FREQ']

        for i in range(0,len(gaps['start'])):
            c = "NC_001781.1"
            s = gaps.loc[i,'start']
            e = gaps.loc[i,'end']
            ivars = ivars[(ivars['POS']<s) | (ivars['POS']>e)]

        ivars = ivars[(ivars['MAF'] > minmaf)]

        smaf = pd.DataFrame({'MAF':ivars['MAF'],
                     'sample':[sample] * len(ivars['MAF']),
                     'target':[target] * len(ivars['MAF'])})

        allmaf = pd.concat([allmaf,smaf])


In [131]:
sumstats = pd.merge(allmaf.groupby(['sample','target'])['MAF'].std().reset_index().rename(columns={'MAF':"MAFsd"}),
                    allmaf.groupby(['sample','target'])['MAF'].mean().reset_index().rename(columns={'MAF':"MAFmean"}))
sumstats

,sample,target,MAFsd,MAFmean
0,Yale-RSV-0001,RSVA,0.134112,0.270450
1,Yale-RSV-0001,RSVB,0.126980,0.236142
2,Yale-RSV-0002,RSVA,0.127935,0.292227
3,Yale-RSV-0002,RSVB,0.130490,0.306642
4,Yale-RSV-0003,RSVA,0.119857,0.280952
5,Yale-RSV-0003,RSVB,0.116475,0.281806


In [124]:
sumstats.to_csv(outsum, index=False,sep="\t")
allmaf.to_csv(outmaf, index=False,sep="\t")